# Mamba SOH — Training at L=4096 on Kaggle GPU

Train MambaSOHPredictor with 4096-token windows on Kaggle P100/T4 GPU.

**Before running:** Add dataset `nasa-battery-dataset` via `+ Add Data`.

Dataset must contain: `cleaned_dataset/metadata.csv` and `cleaned_dataset/data/*.csv`

## Cell 1 — GPU Check

In [ ]:
!nvidia-smi

import torch
print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    vram = torch.cuda.get_device_properties(0).total_memory // 1024**3
    print(f'VRAM: {vram} GB')
else:
    print('WARNING: No GPU detected. Go to Settings -> Accelerator -> GPU P100')

## Cell 2 — Clone Repo

Change `GITHUB_TOKEN` secret name if different, or use public clone below.

In [ ]:
import subprocess, os

BRANCH = 'feat/spectral_kurtosis'
REPO   = '/kaggle/working/ai-module'

# --- Option A: Public repo ---
# subprocess.run([
#     'git', 'clone', '--branch', BRANCH, '--single-branch',
#     'https://github.com/GSU26SE55/ai-module.git', REPO
# ], check=True)

# --- Option B: Private repo (uses Kaggle Secret) ---
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret('GITHUB_TOKEN')
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        f'https://{token}@github.com/GSU26SE55/ai-module.git', REPO
    ], check=True)
    # Remove token from remote URL
    subprocess.run([
        'git', '-C', REPO, 'remote', 'set-url', 'origin',
        'https://github.com/GSU26SE55/ai-module.git'
    ], check=True)
    print('Cloned with token (token cleared from remote)')
except Exception as e:
    print(f'Secret not found ({e}), trying public clone...')
    subprocess.run([
        'git', 'clone', '--branch', BRANCH, '--single-branch',
        'https://github.com/GSU26SE55/ai-module.git', REPO
    ], check=True)

print('Branch:', subprocess.check_output(['git', '-C', REPO, 'branch', '--show-current']).decode().strip())
print('Commit:', subprocess.check_output(['git', '-C', REPO, 'log', '-1', '--oneline']).decode().strip())

## Cell 3 — Install Dependencies

In [ ]:
%pip install -q scipy scikit-learn

import scipy, sklearn
print('scipy:', scipy.__version__)
print('sklearn:', sklearn.__version__)

## Cell 4 — Path Setup & Dataset Check

In [ ]:
import os, sys, shutil

REPO      = '/kaggle/working/ai-module'
PROCESSED = '/kaggle/working/processed'
WEIGHTS   = '/kaggle/working/weights'
LOGS      = '/kaggle/working/logs'

# Auto-detect dataset path
for candidate in [
    '/kaggle/input/nasa-battery-dataset/cleaned_dataset',
    '/kaggle/input/nasa-battery-dataset',
]:
    if os.path.isfile(f'{candidate}/metadata.csv'):
        DATASET = candidate
        break
else:
    # Search
    result = subprocess.check_output(['find', '/kaggle/input', '-name', 'metadata.csv']).decode().strip()
    if result:
        DATASET = os.path.dirname(result.split('\n')[0])
    else:
        raise FileNotFoundError('metadata.csv not found. Add nasa-battery-dataset via + Add Data')

for path in [PROCESSED, WEIGHTS, LOGS]:
    os.makedirs(path, exist_ok=True)

# Symlink repo weights -> WEIGHTS so training artifacts persist in /kaggle/working
repo_weights = f'{REPO}/models/weights'
if os.path.islink(repo_weights):
    os.unlink(repo_weights)
elif os.path.isdir(repo_weights):
    # Copy existing artifacts to WEIGHTS first
    for f in os.listdir(repo_weights):
        src = os.path.join(repo_weights, f)
        dst = os.path.join(WEIGHTS, f)
        if os.path.isfile(src) and not os.path.exists(dst):
            shutil.copy2(src, dst)
    shutil.rmtree(repo_weights)
os.makedirs(os.path.dirname(repo_weights), exist_ok=True)
os.symlink(WEIGHTS, repo_weights)

sys.path.insert(0, REPO)

print(f'REPO:      {REPO}')
print(f'DATASET:   {DATASET}')
print(f'PROCESSED: {PROCESSED}')
print(f'WEIGHTS:   {WEIGHTS}')
print()
print('metadata.csv:', os.path.isfile(f'{DATASET}/metadata.csv'))
data_dir = f'{DATASET}/data'
print('data/:', os.path.isdir(data_dir))
if os.path.isdir(data_dir):
    csvs = [f for f in os.listdir(data_dir) if f.endswith('.csv')]
    print(f'CSV count: {len(csvs)}')

## Cell 5 — Update Config to L=4096

In [ ]:
config_path = f'{REPO}/src/core/config.py'

with open(config_path) as f:
    content = f.read()

# Patch config for L=4096 training
replacements = [
    ('MODEL_VERSION = "1.1"', 'MODEL_VERSION = "1.3"'),
    ('MODEL_VERSION = "1.2"', 'MODEL_VERSION = "1.3"'),
    ('WINDOW_SIZE = 30',      'WINDOW_SIZE = 4096'),
    ('WINDOW_STRIDE = 30',    'WINDOW_STRIDE = 100'),
    ('ISO_FOREST_PATH = os.path.join(WEIGHTS_DIR, "isolation_forest_v1.0.pkl")',
     'ISO_FOREST_PATH = os.path.join(WEIGHTS_DIR, "isolation_forest_v1.3.pkl")'),
    ('ISO_FOREST_PATH = os.path.join(WEIGHTS_DIR, "isolation_forest_v1.1.pkl")',
     'ISO_FOREST_PATH = os.path.join(WEIGHTS_DIR, "isolation_forest_v1.3.pkl")'),
    ('ISO_FOREST_PATH = os.path.join(WEIGHTS_DIR, "isolation_forest_v1.2.pkl")',
     'ISO_FOREST_PATH = os.path.join(WEIGHTS_DIR, "isolation_forest_v1.3.pkl")'),
]
for old, new in replacements:
    content = content.replace(old, new)

with open(config_path, 'w') as f:
    f.write(content)

# Reload config
import importlib
import src.core.config as cfg
importlib.reload(cfg)

print(f'WINDOW_SIZE   = {cfg.WINDOW_SIZE}')
print(f'WINDOW_STRIDE = {cfg.WINDOW_STRIDE}')
print(f'MODEL_VERSION = {cfg.MODEL_VERSION}')
print(f'MAMBA_PATH    = {cfg.MAMBA_PATH}')
print(f'ISO_PATH      = {cfg.ISO_FOREST_PATH}')
assert cfg.WINDOW_SIZE == 4096, 'Config patch failed!'

## Cell 6 — Preprocess

Skip if `/kaggle/working/processed/train.pt` already exists with correct shape.

In [ ]:
import torch

# Check if processed data already exists
need_preprocess = True
train_pt = f'{PROCESSED}/train.pt'
if os.path.isfile(train_pt):
    d = torch.load(train_pt, weights_only=False)
    if d['X'].shape[1] == 4096:
        print(f'Processed data exists with correct L=4096. Skipping preprocess.')
        print(f'Train: {d["X"].shape}')
        need_preprocess = False
    else:
        print(f'Old data shape {d["X"].shape[1]} != 4096. Re-preprocessing...')

if need_preprocess:
    os.chdir(REPO)
    !python scripts/preprocess.py \
        --data-dir "{DATASET}" \
        --output-dir "{PROCESSED}"

## Cell 7 — Verify Data Shape

In [ ]:
import torch

for name in ['train.pt', 'val.pt', 'test.pt']:
    d = torch.load(f'{PROCESSED}/{name}', weights_only=False)
    x_shape  = tuple(d['X'].shape)
    xf_shape = tuple(d['X_feat'].shape)
    y_range  = (float(d['y'].min()), float(d['y'].max()))
    print(f'{name}: X={x_shape}, X_feat={xf_shape}, y=[{y_range[0]:.1f},{y_range[1]:.1f}]')
    assert x_shape[1] == 4096, f'Wrong window size: {x_shape[1]}'
    assert xf_shape[1] == 54,  f'Wrong feat dim: {xf_shape[1]}'

print('\nAll shapes OK')

## Cell 8 — Add GPU Support to train.py

The simple train.py runs on CPU only. This cell patches it to use GPU automatically.

In [ ]:
train_path = f'{REPO}/scripts/train.py'

with open(train_path) as f:
    code = f.read()

# Check if GPU support already added
if 'device = torch.device' not in code:
    # Add device after model creation
    code = code.replace(
        'torch.manual_seed(SEED)\n    model     = MambaSOHPredictor(',
        'device    = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n'
        '    logger.info(f"Device: {device}")\n'
        '    torch.manual_seed(SEED)\n    model     = MambaSOHPredictor('
    )
    # Move model to device
    code = code.replace(
        'model     = MambaSOHPredictor(input_features=INPUT_FEATURES, feat_dim=SPECTRAL_FEAT_DIM, d_model=D_MODEL, d_state=D_STATE)',
        'model     = MambaSOHPredictor(input_features=INPUT_FEATURES, feat_dim=SPECTRAL_FEAT_DIM, d_model=D_MODEL, d_state=D_STATE).to(device)'
    )
    # Move batches to device in training loop
    code = code.replace(
        '        for X_batch, X_feat_batch, y_batch in train_loader:\n            optimizer.zero_grad()\n            pred',
        '        for X_batch, X_feat_batch, y_batch in train_loader:\n'
        '            X_batch, X_feat_batch, y_batch = X_batch.to(device), X_feat_batch.to(device), y_batch.to(device)\n'
        '            optimizer.zero_grad()\n            pred'
    )
    # Move val/test tensors to device in evaluate calls
    code = code.replace(
        'val_pred = model(X_val, X_feat_val)',
        'val_pred = model(X_val.to(device), X_feat_val.to(device))'
    )
    code = code.replace(
        'test_metrics = evaluate(model, X_test, X_feat_test, y_test)',
        'test_metrics = evaluate(model, X_test.to(device), X_feat_test.to(device), y_test.to(device))'
    )
    # Fix evaluate to handle GPU tensors
    code = code.replace(
        '        pred = model(X, X_feat) * 100.0\n        mae  = torch.mean(torch.abs(pred - y)).item()\n        rmse = torch.sqrt(torch.mean((pred - y) ** 2)).item()',
        '        pred = model(X, X_feat) * 100.0\n        y_dev = y.to(pred.device)\n        mae  = torch.mean(torch.abs(pred - y_dev)).item()\n        rmse = torch.sqrt(torch.mean((pred - y_dev) ** 2)).item()',
    )

    with open(train_path, 'w') as f:
        f.write(code)
    print('GPU support patched into train.py')
else:
    print('train.py already has GPU support')

## Cell 9 — Smoke Test (1 epoch, fast)

Verify no errors before full training.

In [ ]:
os.chdir(REPO)
!python scripts/train.py \
    --data-dir "{PROCESSED}" \
    --epochs 1 \
    --log-dir "{LOGS}/smoke"

print('\nSmoke test done. If no error above, proceed to full training.')

## Cell 10 — Full Training

Estimated time on P100: ~3-5 min/epoch, ~3-5 hours total (early stop at 40-70 epochs).

In [ ]:
os.chdir(REPO)
!python scripts/train.py \
    --data-dir "{PROCESSED}" \
    --epochs 150 \
    --log-dir "{LOGS}"

## Cell 11 — Verify Artifacts

In [ ]:
required = [
    'scaler.pkl',
    'feature_scaler.pkl',
    'soh_mamba_v1.3.pth',
    'isolation_forest_v1.3.pkl',
]

all_ok = True
for fname in required:
    path = f'{WEIGHTS}/{fname}'
    ok   = os.path.isfile(path)
    size = os.path.getsize(path) / 1024 if ok else 0
    print(f'{fname}: {"OK" if ok else "MISSING"}  ({size:.0f} KB)')
    all_ok = all_ok and ok

print()
print('All artifacts ready:', all_ok)

## Cell 12 — Check Training Results

In [ ]:
import glob

log_files = glob.glob(f'{LOGS}/**/train_*.log', recursive=True)
if log_files:
    latest = max(log_files, key=os.path.getmtime)
    print(f'Log: {latest}')
    !grep -E 'Test MAE|Test RMSE|ACHIEVED|WARNING.*target' "{latest}"
else:
    print('No log found')

## Cell 13 — Package for Download

In [ ]:
import shutil, torch

# Check model checkpoint
ckpt_path = f'{WEIGHTS}/soh_mamba_v1.3.pth'
if os.path.isfile(ckpt_path):
    ck = torch.load(ckpt_path, map_location='cpu', weights_only=False)
    print(f'Model v{ck["version"]}:')
    print(f'  window_size   = {ck["window_size"]}')
    print(f'  input_features= {ck["input_features"]}')
    print(f'  feat_dim      = {ck["feat_dim"]}')
    print(f'  d_model       = {ck.get("d_model", 64)}')
    print(f'  Test MAE      = {ck["test_mae"]}%')
    print(f'  Test RMSE     = {ck["test_rmse"]}%')

# Create zip for download
out_zip = '/kaggle/working/mamba_v1.3_L4096_artifacts'
shutil.make_archive(out_zip, 'zip', WEIGHTS)
zip_size = os.path.getsize(f'{out_zip}.zip') / 1024
print(f'\nCreated: {out_zip}.zip  ({zip_size:.0f} KB)')
print('Download from: Kaggle Output tab -> mamba_v1.3_L4096_artifacts.zip')

## Cell 14 — View Training Curve

In [ ]:
import re, glob
import matplotlib.pyplot as plt

log_files = glob.glob(f'{LOGS}/**/train_*.log', recursive=True)
if not log_files:
    print('No log found')
else:
    latest = max(log_files, key=os.path.getmtime)
    epochs, val_maes, train_losses = [], [], []
    with open(latest) as f:
        for line in f:
            m = re.search(r'DEBUG\s+(\d+)\s+([\d.]+)\s+[\d.]+\s+([\d.]+)', line)
            if m:
                epochs.append(int(m.group(1)))
                train_losses.append(float(m.group(2)))
                val_maes.append(float(m.group(3)))

    if epochs:
        fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))
        ax1.plot(epochs, train_losses, label='Train Loss')
        ax1.set_xlabel('Epoch'); ax1.set_ylabel('Loss')
        ax1.set_title('Training Loss'); ax1.legend()
        ax2.plot(epochs, val_maes, label='Val MAE%', color='orange')
        ax2.axhline(2.0, color='red', linestyle='--', label='Target 2%')
        ax2.set_xlabel('Epoch'); ax2.set_ylabel('MAE %')
        ax2.set_title('Validation MAE'); ax2.legend()
        plt.tight_layout()
        plt.savefig('/kaggle/working/training_curve.png', dpi=150)
        plt.show()
        print(f'Best Val MAE: {min(val_maes):.4f}% at epoch {epochs[val_maes.index(min(val_maes))]}')
    else:
        print('No epoch data in log yet')